# Predictive Circuit Coding Training

This notebook uses the shared Drive dataset as a read-only source, keeps artifacts local during the run for reliability, and exports the finished run to a clearly separate Drive folder when training is done.

Default behavior is simple: train on a small notebook-only familiar-session subset so debug runs stay fast and coherent.

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil
from datetime import datetime

drive.mount('/content/drive')
repo_root = Path('/content/predictive_circuit_coding')
github_repo_url = 'https://github.com/jacobposchl/predictive_circuit_coding'

# Shared folder from the storage-heavy Drive account: read dataset from here.
shared_drive_root = Path('/content/drive/MyDrive/pcc_runs')
drive_data_root = shared_drive_root / 'data' / 'allen_visual_behavior_neuropixels'

# Keep exported Colab outputs in a clearly different folder name so they do not get
# confused with the source dataset bundle.
drive_export_root = Path('/content/drive/MyDrive/pcc_colab_outputs')
drive_export_root.mkdir(parents=True, exist_ok=True)

print('Repo root:', repo_root)
print('GitHub repo:', github_repo_url)
print('Shared dataset root:', drive_data_root)
print('Drive export root:', drive_export_root)

In [ ]:
if not repo_root.exists():
    !git clone {github_repo_url} {repo_root}
%cd {repo_root}
# Keep Colab installs minimal to avoid restart prompts from preloaded widget packages.
!pip install -e .

In [ ]:
repo_dataset_root = repo_root / 'data' / 'allen_visual_behavior_neuropixels'
repo_artifacts_root = repo_root / 'artifacts'

assert drive_data_root.exists(), f'Missing Drive dataset bundle: {drive_data_root}'

repo_dataset_root.parent.mkdir(parents=True, exist_ok=True)
if repo_dataset_root.exists() or repo_dataset_root.is_symlink():
    if repo_dataset_root.is_symlink():
        repo_dataset_root.unlink()
    else:
        shutil.rmtree(repo_dataset_root)
repo_dataset_root.symlink_to(drive_data_root, target_is_directory=True)

if repo_artifacts_root.exists():
    shutil.rmtree(repo_artifacts_root)
repo_artifacts_root.mkdir(parents=True, exist_ok=True)

print('Linked dataset into repo:', repo_dataset_root)
print('Using local artifact directory:', repo_artifacts_root)

In [ ]:
from predictive_circuit_coding.utils import (
    NotebookDatasetConfig,
    NotebookStageReporter,
    prepare_notebook_runtime_context,
    resolve_notebook_checkpoint,
    run_streaming_command,
    verify_paths_exist,
)

NOTEBOOK_STEP_LOG_EVERY = 16
NOTEBOOK_DATASET = NotebookDatasetConfig(
    use_full_dataset=False,
    experience_level='Familiar',
    max_sessions=10,
)

reporter = NotebookStageReporter(name='colab-train', expected_duration='2-10 minutes for debug runs, longer for full A100 runs')
reporter.banner('Predictive Circuit Coding Training', subtitle='Setup, preflight, runtime selection, training, evaluation, and export')

base_experiment_config = repo_root / 'configs' / 'pcc' / 'predictive_circuit_coding_base.yaml'
data_config = repo_root / 'configs' / 'pcc' / 'allen_visual_behavior_neuropixels_local.yaml'
runtime_context = prepare_notebook_runtime_context(
    base_experiment_config=base_experiment_config,
    runtime_experiment_config=repo_root / 'colab_runtime_experiment.yaml',
    session_catalog_csv=repo_root / 'data' / 'allen_visual_behavior_neuropixels' / 'manifests' / 'session_catalog.csv',
    artifact_root=repo_root / 'artifacts',
    dataset_config=NOTEBOOK_DATASET,
    step_log_every=NOTEBOOK_STEP_LOG_EVERY,
)
experiment_config = runtime_context.experiment_config_path
checkpoint_dir = runtime_context.checkpoint_dir
summary_path = runtime_context.summary_path
resume_checkpoint = runtime_context.checkpoint_path
dataset_selection_active = runtime_context.dataset_selection_active

run_label = datetime.now().strftime('train_run_%Y%m%d_%H%M%S')
drive_run_export_root = drive_export_root / run_label

print('Experiment config:', experiment_config)
print('Data config:', data_config)
print('Runtime selection active:', dataset_selection_active)
if not NOTEBOOK_DATASET.use_full_dataset:
    print('Target experience level:', NOTEBOOK_DATASET.experience_level)
    print('Max sessions:', NOTEBOOK_DATASET.max_sessions)
    print('Selected sessions:', runtime_context.selected_session_count)
print('Notebook step-log cadence:', NOTEBOOK_STEP_LOG_EVERY)
print('Selection output name:', runtime_context.selection_output_name)
print('Resolved checkpoint dir:', checkpoint_dir.resolve())
print('Resolved summary path:', summary_path.resolve())
print('Checkpoint path:', resume_checkpoint)
print('Notebook profile:', runtime_context.profile_path)
print('Drive export path:', drive_run_export_root)

In [ ]:
reporter.begin('preflight', next_artifact='best checkpoint + evaluation summary')
paths_ok = verify_paths_exist({
    'experiment_config': experiment_config,
    'data_config': data_config,
    'drive_dataset_root': drive_data_root,
})
print(paths_ok)
assert all(paths_ok.values()), 'Missing config files or dataset bundle for training notebook.'

if dataset_selection_active:
    reporter.begin('runtime selection', next_artifact='selected split manifest')
    run_streaming_command([
        'pcc-prepare-data',
        'materialize-runtime-selection',
        '--config', str(experiment_config),
        '--data-config', str(data_config),
    ], cwd=repo_root, step_log_every=NOTEBOOK_STEP_LOG_EVERY)
    reporter.finish('runtime selection')
else:
    print('Using the full canonical dataset. No runtime subset will be materialized.')

reporter.finish('preflight')

In [ ]:
reporter.begin('training', next_artifact='local checkpoint under artifacts/checkpoints')
%cd {repo_root}
run_streaming_command([
    'pcc-train',
    '--config', str(experiment_config),
    '--data-config', str(data_config),
], cwd=repo_root, step_log_every=NOTEBOOK_STEP_LOG_EVERY)
resume_checkpoint = resolve_notebook_checkpoint(summary_path=summary_path, checkpoint_dir=checkpoint_dir)
if not resume_checkpoint.exists():
    checkpoint_candidates = sorted(checkpoint_dir.glob('*.pt'))
    print('Available checkpoints:', [path.name for path in checkpoint_candidates])
    raise AssertionError(f'Expected checkpoint was not written: {resume_checkpoint}')
reporter.note_checkpoint(resume_checkpoint)
reporter.finish('training')

In [ ]:
reporter.begin('evaluation', next_artifact='local evaluation summary json')
%cd {repo_root}
run_streaming_command([
    'pcc-evaluate',
    '--config', str(experiment_config),
    '--data-config', str(data_config),
    '--checkpoint', str(resume_checkpoint),
    '--split', 'valid',
], cwd=repo_root, step_log_every=NOTEBOOK_STEP_LOG_EVERY)
reporter.finish('evaluation')

In [ ]:
reporter.begin('export', next_artifact='Drive copy of local artifacts')
if drive_run_export_root.exists():
    shutil.rmtree(drive_run_export_root)
shutil.copytree(repo_artifacts_root, drive_run_export_root)
print('Exported local artifacts to:', drive_run_export_root)
reporter.finish('export')